# Linearization


In [1]:
import sympy as sp

from lib.parameters import (
    FR_0,
    T1_0,
    T2_0,
    T3_0,
    Ff1_0,
    Ff2_0,
    xA1_0,
    xA2_0,
    xA3_0,
    xB1_0,
    xB2_0,
    xB3_0,
)
from lib.sp_model import (
    FR,
    T1,
    T2,
    T3,
    Ff1,
    Ff2,
    dT1dt,
    dT2dt,
    dT3dt,
    dxA1dt,
    dxA2dt,
    dxA3dt,
    dxB1dt,
    dxB2dt,
    dxB3dt,
    xA1,
    xA2,
    xA3,
    xB1,
    xB2,
    xB3,
)

f = sp.Matrix([dT1dt, dT2dt, dT3dt, dxA1dt, dxB1dt, dxA2dt, dxB2dt, dxA3dt, dxB3dt])
X = sp.Matrix([T1, T2, T3, xA1, xB1, xA2, xB2, xA3, xB3])
U = sp.Matrix([Ff1, Ff2, FR])

U0_vals = {Ff1: Ff1_0, Ff2: Ff2_0, FR: FR_0}

X0_vals = {
    T1: T1_0,
    T2: T2_0,
    T3: T3_0,
    xA1: xA1_0,
    xB1: xB1_0,
    xA2: xA2_0,
    xB2: xB2_0,
    xA3: xA3_0,
    xB3: xB3_0,
}

A = f.jacobian(X).subs(X0_vals).subs(U0_vals)
B = f.jacobian(U).subs(X0_vals).subs(U0_vals)


# Model


In [2]:
from collections.abc import Callable

import numpy as np


def model(t: float, y: np.ndarray, u: list[Callable[[float], float]]):
    X = y
    U = np.array([ui(t) for ui in u]) - np.array(list(U0_vals.values()))

    return A @ X + B @ U


# Simulation


In [3]:
from scipy.integrate import solve_ivp

from lib.inputs import Ff1_func, Ff2_func, FR_func
from lib.parameters import y0
from lib.plots import plot_system_inputs, plot_system_outputs
from lib.utils import input_names

base_funcs = [Ff1_func, Ff2_func, FR_func]

t = np.linspace(0, 5, 1500)

for idx, name in enumerate(input_names):
    u = list(base_funcs)
    u[idx] = lambda t, f=base_funcs[idx]: f(t, disturb=True)  # type: ignore

    sol = solve_ivp(
        model,
        [t[0], t[-1]],
        np.zeros_like(y0),
        t_eval=t,
        args=(u,),
        rtol=1e-10,
        atol=1e-10,
    )

    u_eval = [ui(t) for ui in u]
    soly = sol.y + np.array(list(X0_vals.values()))[:, None]
    plot_system_inputs(t, u_eval, f"linear/pulse-{name}-inputs")
    plot_system_outputs(t, soly, f"linear/pulse-{name}-outputs")


Plot saved to ../figures/linear/pulse-Ff1-inputs.png
Plot saved to ../figures/linear/pulse-Ff1-outputs.png
Plot saved to ../figures/linear/pulse-Ff2-inputs.png
Plot saved to ../figures/linear/pulse-Ff2-outputs.png
Plot saved to ../figures/linear/pulse-FR-inputs.png
Plot saved to ../figures/linear/pulse-FR-outputs.png


In [4]:
from lib.np_model import model as nonlinear_model
from lib.plots import plot_or_show, plt
from lib.utils import (
    input_labels,
    input_units,
    output_labels,
    output_names,
    output_units,
)


def Ff1_custom(t):
    if t < 0.5:
        return Ff1_0
    elif t < 2.0:
        return Ff1_0 + 1.0
    elif t < 4.0:
        return Ff1_0 + 3.0
    return Ff1_0


u = [Ff1_custom, Ff2_func, FR_func]

t = np.linspace(0, 6.5, 3000)

sol_nl = solve_ivp(
    nonlinear_model, [t[0], t[-1]], y0, t_eval=t, args=(u,), rtol=1e-10, atol=1e-10
)

sol_lin = solve_ivp(
    model, [t[0], t[-1]], np.zeros_like(y0), t_eval=t, args=(u,), rtol=1e-10, atol=1e-10
)

Y_lin = sol_lin.y + np.array(list(X0_vals.values()))[:, None]

# --- Plotting ---
input_id = 0
output_id = 4
G_name = f"{output_names[output_id]}_{input_names[input_id]}"
G_ylabel = f"{output_labels[output_id]} / {output_units[output_id]}"

Ff1_plot = np.array([Ff1_custom(tt) for tt in t])

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(8, 6))

# Entrada: perturbação em Ff1
ax[0].plot(t, Ff1_plot)
ax[0].set_ylabel(f"{input_labels[input_id]} / {input_units[input_id]}")
ax[0].grid(True)

# Saída
ax[1].plot(sol_nl.t, sol_nl.y[output_id], label="Não linear")
ax[1].plot(sol_lin.t, Y_lin[output_id], "--", label="Linear")

ax[1].set_xlabel("Tempo / h")
ax[1].set_ylabel(G_ylabel)
ax[1].grid(True)
ax[1].legend()

plot_or_show(f"linear_vs_nonlinear/{G_name}")


Plot saved to ../figures/linear_vs_nonlinear/xB1_Ff1.png


# Export A B C Matrix


In [5]:
from pathlib import Path

OUT_DIR = Path("../LaTeX/common")  # ajuste para a pasta dos .tex


def fmt(x, sig=3, tol=1e-12):
    """Formata um número em LaTeX com `sig` algarismos significativos."""
    if abs(x) < tol:
        return "0"
    s = f"{x:.{sig - 1}e}"  # ex.: '-1.54e+03'
    mant, exp = s.split("e")
    exp = int(exp)
    mant = mant.replace(".", "{,}")  # vírgula decimal
    if -2 <= exp <= 2:  # números "normais" sem notação científica
        return f"{x:.{max(sig - 1 - exp, 0)}f}".replace(".", "{,}")
    return f"{mant} \\cdot 10^{{{exp}}}"


def matrix_to_latex(M, sig=3):
    M = (
        np.array(M.tolist(), dtype=float)
        if hasattr(M, "tolist")
        else np.array(M, dtype=float)
    )
    rows = [" & ".join(fmt(v, sig) for v in row) for row in M]
    return "\\begin{bmatrix}\n" + " \\\\\n".join(rows) + "\n\\end{bmatrix}"


def save_matrix(M, name, label, filename, sig=3, resize=None):
    """Salva a matriz em um .tex com ambiente equation e label.

    resize: fração da largura (ex.: 0.95) para ajustar matrizes largas com \\resizebox.
    """
    body = matrix_to_latex(M, sig)
    if resize:
        content = (
            "\\begin{equation}\n"
            f"  \\label{{{label}}}\n"
            f"  \\resizebox{{{resize}\\linewidth}}{{!}}{{$\\displaystyle \\mathbf{{{name}}} = {body}$}}\n"
            "\\end{equation}\n"
        )
    else:
        content = (
            "\\begin{equation}\n"
            f"  \\label{{{label}}}\n"
            f"  \\mathbf{{{name}}} = {body}\n"
            "\\end{equation}\n"
        )
    (OUT_DIR / filename).write_text(content, encoding="utf-8")


C = np.eye(A.shape[0])  # estados = saídas

save_matrix(A, "A", "eq:matriz_A", "A_matrix.tex", sig=3, resize=0.9)
save_matrix(B, "B", "eq:matriz_B", "B_matrix.tex", sig=3)
save_matrix(C, "C", "eq:matriz_C", "C_matrix.tex", sig=1)


In [6]:
import hickle as hkl

OUT = Path("../outputs")

# converte de sympy para numpy (float)
A_np = np.array(A.tolist(), dtype=float)
B_np = np.array(B.tolist(), dtype=float)
C_np = np.eye(A_np.shape[0])

hkl.dump(A_np, OUT / "A.hkl")
hkl.dump(B_np, OUT / "B.hkl")
hkl.dump(C_np, OUT / "C.hkl")
